In [1]:
import numpy as np
import random
import math

In [2]:
def is_feasible(sol):
    return sum(sol) >= 1

In [3]:
def init_sol_fixed(num_resources):
    return np.ones(num_resources) # pravimo fiksno početno rešenje

In [4]:
def init_sol_random(num_resources, p = 0.25):
    sol = np.array([random.random() < p for _ in range(num_resources)]) # nasumično biramo početno rešenje
    
    if not is_feasible(sol):
        sol[random.randrange(num_resources)] = True
    return sol

In [5]:
def calc_sol_value(sol, costs, facility_cost):
    total_cost = sol @ facility_cost
    
    """
    for i in range(len(costs)):
        curr_min = float('inf')
        for j in range(len(costs[i])):
            if sol[j] and costs[i][j] < curr_min:  
                curr_min = costs[i][j]
        total_cost += curr_min
    """
    # ekvivalentno, ali dosta brže rešenje :)
    built = np.where(sol)[0]
    total_cost += costs[:, built].min(axis = 1).sum()
    
    return total_cost

In [6]:
def flip_one(sol):
    n = random.randrange(len(sol))
    sol[n] = not sol[n]
    
    # nepotrebno ako cenu računamo koristeći dvostruku petlju, ali neophodno ako koristimo pogodnosti numpy paketa
    if not is_feasible(sol):
        sol[n] = not sol[n]
        return flip_one(sol)
    
    return n

In [7]:
def local_search(num_users, num_resources, costs, facility_cost, num_iters):
    sol = init_sol_fixed(num_resources)
    curr_value = calc_sol_value(sol, costs, facility_cost)
    
    for _ in range(num_iters):
        n = flip_one(sol)
        
        new_value = calc_sol_value(sol, costs, facility_cost)
        
        if new_value < curr_value:
            curr_value = new_value
        else:
            sol[n] = not sol[n]
    return sol, curr_value

In [8]:
import sys
sys.path.append("..")

from uflp_utils import read_instance, read_bk_instance

In [9]:
num_users, num_resources, costs, facility_cost = read_instance("../cas2/uflp1.txt")
num_users, num_resources, costs, facility_cost

(3,
 3,
 array([[ 1, 12,  3],
        [ 2,  7, 41],
        [19, 21,  7]]),
 array([12, 11, 13]))

In [10]:
for _ in range(10):
    print(local_search(num_users, num_resources, costs, facility_cost, 10000))
# optimalna vrednost za ovu instancu je 34

(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([0., 1., 1.]), np.float64(41.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([0., 1., 1.]), np.float64(41.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([0., 1., 1.]), np.float64(41.0))


In [11]:
num_users_bk, num_resources_bk, costs_bk, facility_cost_bk = read_bk_instance("../BildeKrarup/B/B1.1")
num_users_bk, num_resources_bk

(100, 50)

In [12]:
for _ in range(10):
    print(local_search(num_users_bk, num_resources_bk, costs_bk, facility_cost_bk, 10000))
# optimalna vrednost za ovu instancu je 23468

(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(23468.0))
(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(23468.0))
(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(23932.0))
(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(23932.0))
(array([0., 0., 0., 

In [13]:
def local_search_fst_improvement(num_users, num_resources, costs, facility_cost):
    sol = init_sol_random(num_resources) # kada bismo pravili fiksno početno rešenje, uvek bismo prolazili kroz 
                                        # iste korake i na kraju dobili uvek isto rešenje
    curr_value = calc_sol_value(sol, costs, facility_cost)
    
    while True:
        first_value = curr_value 
        
        for i in range(len(sol)):
            sol[i] = not sol[i]
            
            # sledeći uslov bi bio nepotreban ako bi se vrednost rešenja računala dvostrukom petljom
            if not is_feasible(sol):
                sol[i] = not sol[i] # u slučaju nevalidnog rešenja, treba vratiti obrnuti bit pre sledeće promene
                continue
            
            new_value = calc_sol_value(sol, costs, facility_cost)
            
            if new_value < curr_value:
                curr_value = new_value
                break
            else:
                sol[i] = not sol[i]
        if first_value == curr_value:
            break
    return sol, curr_value 

In [14]:
for _ in range(10):
    print(local_search_fst_improvement(num_users, num_resources, costs, facility_cost))

(array([ True, False, False]), np.int64(34))
(array([False,  True,  True]), np.int64(41))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))


In [15]:
for _ in range(10):
    print(local_search_fst_improvement(num_users_bk, num_resources_bk, costs_bk, facility_cost_bk))

(array([False, False, False, False, False, False, False, False, False,
       False,  True, False, False,  True, False, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False,  True, False, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False, False]), np.int64(23932))
(array([False, False, False, False, False, False, False, False, False,
       False,  True, False, False,  True,  True, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False,  True, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False, False]), np.int64(23468))
(array([False, False, False, False, False, False, False, False, False,
       False,  True, False, False,  True, False, False, False, False,
       False, False, False, False, 

In [16]:
# za domaći:
# local_search_best_improvement
# da li bi swap dao bolje rešenje nego flip?
# kako se local_search ponaša ako se početno rešenje inicijalizuje nasumično?

In [17]:
def simulated_annealing(num_users, num_resources, costs, facility_cost, num_iters, T = 100, alpha = 0.95):
    sol = init_sol_fixed(num_resources)
    curr_value = calc_sol_value(sol, costs, facility_cost)
    
    best_value = curr_value
    best_sol = sol.copy()
    
    for it in range(num_iters):
        n = flip_one(sol)
        
        new_value = calc_sol_value(sol, costs, facility_cost)
        
        if new_value < curr_value:
            curr_value = new_value
            if new_value < best_value:
                best_value = new_value
                best_sol = sol.copy()
        else:
            p = math.exp(-(new_value - curr_value) / T)
            
            # kao verovatnoću da prihvatimo lošije rešenje možemo uzeti bilo koju funkciju koja ravnomerno opada, npr.
            # p = 1 / (it + 1)
            # p = 1 / math.sqrt(it + 1) 

            if random.random() < p:
                curr_value = new_value
            else:
                sol[n] = not sol[n]
        T = T * alpha
    return best_sol, best_value

In [18]:
for _ in range(10):
    print(simulated_annealing(num_users, num_resources, costs, facility_cost, 1000))

(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))


In [19]:
for _ in range(10):
    print(simulated_annealing(num_users_bk, num_resources_bk, costs_bk, facility_cost_bk, 1000))

(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(23932.0))
(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(23468.0))
(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(23932.0))
(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(23932.0))
(array([0., 0., 0., 

In [20]:
def flip_k(sol, k):
    bits = random.sample(range(len(sol)), k)
    for bit in bits:
        sol[bit] = not sol[bit]
        
    return bits

In [21]:
def rvns(num_users, num_resources, costs, facility_cost, num_iters, max_level):
    sol = init_sol_fixed(num_resources)
    curr_value = calc_sol_value(sol, costs, facility_cost)
    
    for _ in range(num_iters):
        level = 1
        
        while level <= max_level:
            bits = flip_k(sol, level)
            
            # ponovo, ovaj uslov bi bio nepotreban kada bi se vrednost rešenja računala dvostrukom petljom
            if not is_feasible(sol):
                for bit in bits:
                    sol[bit] = not sol[bit]
                level += 1
                continue
            
            new_value = calc_sol_value(sol, costs, facility_cost)
            
            if new_value < curr_value:
                curr_value = new_value
                level = 1
            else:
                level += 1
                for bit in bits:
                    sol[bit] = not sol[bit]
    return sol, curr_value

In [22]:
for _ in range(10):
    print(rvns(num_users, num_resources, costs, facility_cost, 10, num_resources))

(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))
(array([1., 0., 0.]), np.float64(34.0))


In [23]:
for _ in range(10):
    print(rvns(num_users_bk, num_resources_bk, costs_bk, facility_cost_bk, 10, num_resources_bk))

(array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0.,
       1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(30084.0))
(array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 1., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 1., 0., 0., 0.]), np.float64(38398.0))
(array([1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 1.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
       0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]), np.float64(54237.0))
(array([1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 1., 0., 0.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]), np.float64(30954.0))
(array([0., 0., 0., 

In [24]:
from tsp_utils import read_tsp

In [25]:
data = read_tsp("bays29.tsp")
data

array([[  0, 107, 241, 190, 124,  80, 316,  76, 152, 157, 283, 133, 113,
        297, 228, 129, 348, 276, 188, 150,  65, 341, 184,  67, 221, 169,
        108,  45, 167],
       [107,   0, 148, 137,  88, 127, 336, 183, 134,  95, 254, 180, 101,
        234, 175, 176, 265, 199, 182,  67,  42, 278, 271, 146, 251, 105,
        191, 139,  79],
       [241, 148,   0, 374, 171, 259, 509, 317, 217, 232, 491, 312, 280,
        391, 412, 349, 422, 356, 355, 204, 182, 435, 417, 292, 424, 116,
        337, 273,  77],
       [190, 137, 374,   0, 202, 234, 222, 192, 248,  42, 117, 287,  79,
        107,  38, 121, 152,  86,  68,  70, 137, 151, 239, 135, 137, 242,
        165, 228, 205],
       [124,  88, 171, 202,   0,  61, 392, 202,  46, 160, 319, 112, 163,
        322, 240, 232, 314, 287, 238, 155,  65, 366, 300, 175, 307,  57,
        220, 121,  97],
       [ 80, 127, 259, 234,  61,   0, 386, 141,  72, 167, 351,  55, 157,
        331, 272, 226, 362, 296, 232, 164,  85, 375, 249, 147, 301, 118,
    

In [26]:
def init_tsp(num_nodes):
    sol = list(range(num_nodes))
    random.shuffle(sol)
    
    return sol

In [27]:
def calc_tsp(cost_matrix, sol):
    total_cost = 0
    for i in range(len(sol)- 1):
        total_cost += cost_matrix[sol[i]][sol[i + 1]]
    total_cost += cost_matrix[sol[-1]][sol[0]]
    
    return total_cost

In [28]:
def swap_2(sol):
    i, j = random.sample(range(len(sol)), 2)
    
    tmp = sol[i]
    sol[i] = sol[j]
    sol[j] = tmp
    
    # ekvivalentno:
    # sol[i], sol[j] = sol[j], sol[i]
    
    return i,j

In [29]:
def swap_k(sol, k):
    swaps = []

    for _ in range(k):
        i, j = random.sample(range(len(sol)), 2)

        sol[i], sol[j] = sol[j], sol[i]

        swaps.append((i, j))

    return swaps

In [30]:
def vns_tsp(cost_matrix, num_iters, max_level):
    sol = init_tsp(len(cost_matrix))
    curr_value = calc_tsp(cost_matrix, sol)
    
    for _ in range(num_iters):
        level = 1
        
        while level <= max_level:
            swaps = swap_k(sol, level)
            new_value = calc_tsp(cost_matrix, sol)
            
            if new_value < curr_value:
                curr_value = new_value
                level = 1
            else:
                level += 1
                for i, j in reversed(swaps):
                    sol[i], sol[j] = sol[j], sol[i]

    return sol, curr_value

In [31]:
for _ in range(10):
    print(vns_tsp(data, 10, len(data)))
# najbolje rešenje za ovu instancu je 2020

([3, 9, 28, 2, 1, 6, 22, 16, 14, 21, 10, 24, 12, 0, 11, 5, 4, 25, 20, 7, 26, 18, 23, 8, 27, 19, 13, 17, 15], np.int64(3891))
([26, 7, 16, 24, 22, 12, 9, 23, 5, 11, 20, 3, 14, 17, 19, 8, 4, 2, 28, 27, 0, 25, 1, 6, 10, 21, 13, 18, 15], np.int64(3796))
([23, 8, 25, 20, 22, 27, 26, 3, 17, 16, 10, 15, 24, 6, 18, 5, 11, 7, 9, 12, 4, 21, 13, 14, 0, 19, 2, 28, 1], np.int64(4180))
([2, 26, 16, 17, 1, 19, 20, 11, 9, 6, 18, 21, 13, 10, 14, 3, 4, 8, 0, 27, 22, 7, 23, 12, 24, 15, 5, 25, 28], np.int64(3833))
([21, 13, 8, 15, 22, 7, 26, 20, 12, 0, 4, 19, 3, 27, 5, 11, 1, 28, 2, 25, 18, 23, 10, 16, 24, 6, 9, 14, 17], np.int64(4032))
([24, 16, 21, 13, 18, 1, 4, 19, 2, 23, 11, 0, 28, 25, 10, 6, 22, 15, 12, 14, 17, 9, 20, 7, 27, 5, 8, 26, 3], np.int64(4173))
([27, 26, 22, 6, 19, 0, 12, 23, 9, 16, 13, 24, 3, 1, 28, 21, 14, 17, 5, 8, 4, 11, 20, 2, 25, 10, 18, 15, 7], np.int64(4058))
([0, 26, 1, 19, 16, 6, 15, 24, 10, 5, 11, 17, 18, 21, 13, 14, 12, 25, 28, 2, 9, 8, 3, 23, 22, 4, 20, 27, 7], np.int64(4424))


In [32]:
# domaći:
# isprobati lokalnu pretragu i simulirano kaljenje na TSP problemu
# uporediti kakva se rešenja dobijaju ako se kao izmena trenutnog rešenja koristi: swap_2, swap_range i 
    # premeštanje jednog grada na drugo mesto 

## Dodatno: originalni VNS

In [33]:
def local_search_for_vns(sol, best_value, costs, facility_cost, num_iters):
    curr_value = best_value
    
    for _ in range(num_iters):
        n = flip_one(sol)
        
        new_value = calc_sol_value(sol, costs, facility_cost)
        
        if new_value < curr_value:
            curr_value = new_value
        else:
            sol[n] = not sol[n]
    return sol, curr_value

In [34]:
def vns(num_users, num_resources, costs, facility_cost, num_iters, max_level):
    sol = init_sol_random(num_resources, 0.25)
    curr_value = calc_sol_value(sol, costs, facility_cost)
    
    new_sol = sol.copy()
    
    for _ in range(num_iters):
        level = 1
        
        while level <= max_level:
            bits = flip_k(new_sol, level)
            
            if not is_feasible(new_sol):
                for bit in bits:
                    new_sol[bit] = not new_sol[bit]
                level += 1
                continue
            
            new_value = calc_sol_value(new_sol, costs, facility_cost)
            
            new_sol, new_value = local_search_for_vns(new_sol, new_value, costs, facility_cost, 1000)
            
            if new_value < curr_value:
                curr_value = new_value
                sol = new_sol.copy()
                level = 1
            else:
                new_sol = sol.copy()
                level += 1
    return sol, curr_value

In [35]:
for _ in range(10):
    print(vns(num_users, num_resources, costs, facility_cost, 10, num_resources))

(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))
(array([ True, False, False]), np.int64(34))


In [36]:
for _ in range(10):
    print(vns(num_users_bk, num_resources_bk, costs_bk, facility_cost_bk, 10, 10))

(array([False, False, False, False, False, False, False, False, False,
       False,  True, False, False,  True,  True, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False,  True, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False, False]), np.int64(23468))
(array([False, False, False, False, False, False, False, False, False,
       False,  True, False, False,  True,  True, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False,  True, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False, False]), np.int64(23468))
(array([False, False, False, False, False, False, False, False, False,
       False,  True, False, False,  True,  True, False, False, False,
       False, False, False, False, 